# Conditional DDPM: classifier-based in-sample evaluation
Train a small CNN on 10,000 generated MNIST images (1,000 per digit), then evaluate it on those same images.

**Scope:** this is a diagnostic of how well a CNN can fit the generator's label assignments. It is not held-out accuracy or independent evidence of image quality. Memorization and generator artifacts can produce high accuracy.

Run cells in order using **DDPM (.venv)** on the allocated GPU. Classifier training starts only when you execute its training cell.

## 1. Dataset, inputs, and targets
The DDPM models p(image | digit). Each image begins as independent Gaussian noise and is refined over 1,000 reverse steps. A 10-dimensional one-hot digit vector stays fixed throughout generation.

The DDPM notation uses y for the image and x for its condition. For this classifier we instead name the target **y_target**:
- **Input:** only the 28×28 grayscale generated image, with one channel. The classifier never receives the conditioning vector.
- **Target y_target:** the integer requested digit, 0–9. This becomes the supervised classification target because the generator was asked to produce that class.
- **Output:** ten logits; softmax gives class probabilities and argmax gives the predicted digit.

The target is an assigned conditioning label, not an independently verified annotation. An image may be ambiguous or fail to depict its requested digit.

The dataset contains raw float32 images.npy, integer labels.npy, and metadata.json recording the checkpoint, seed, generation batch size, and completion. Raw image values can exceed the nominal [-1,1] range. We clip them to that display range and scale to [0,1] for classification.

In [ ]:
from pathlib import Path
import json
import time
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from flax import linen as nn
from flax.training import train_state
import optax

PROJECT = Path("/blue/ark007/juntaekwon/projects/DDPM")
DATA_DIR = PROJECT / "samples/conditional/evaluation_step_00080000"
RESULT_DIR = PROJECT / "runs/conditional_classifier_evaluation"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print("Backend:", jax.default_backend(), "Devices:", jax.devices())
metadata = json.loads((DATA_DIR / "metadata.json").read_text())
raw_images = np.load(DATA_DIR / "images.npy")
y_target = np.load(DATA_DIR / "labels.npy").astype(np.int32)
assert raw_images.shape == (10000, 28, 28, 1)
assert y_target.shape == (10000,)
assert np.isfinite(raw_images).all()
assert np.array_equal(np.bincount(y_target, minlength=10), np.full(10, 1000))
images = ((np.clip(raw_images, -1, 1) + 1) / 2).astype(np.float32)
print("Checkpoint:", metadata["checkpoint"])
print("Image shape:", images.shape, "Class counts:", np.bincount(y_target))

## 2. Inspect examples
Rows show requested labels 0–9. Each column contains a separate generated image. Compare the requested digit with the visible handwriting; the assigned target does not change when an image looks incorrect.

In [ ]:
fig, axes = plt.subplots(10, 8, figsize=(8, 10))
for digit in range(10):
    indices = np.flatnonzero(y_target == digit)[:8]
    for column, index in enumerate(indices):
        axes[digit, column].imshow(images[index, :, :, 0], cmap="gray", vmin=0, vmax=1)
        axes[digit, column].set_xticks([])
        axes[digit, column].set_yticks([])
    axes[digit, 0].set_ylabel(str(digit))
fig.suptitle("Rows: requested conditioning digits")
plt.tight_layout()
plt.show()

## 3. CNN and objective
The architecture is deliberately small:
1. 3×3 convolution, 32 channels, ReLU, 2×2 max pooling (28×28 → 14×14).
2. 3×3 convolution, 64 channels, ReLU, 2×2 max pooling (14×14 → 7×7).
3. Flatten, 128-unit dense layer with ReLU, then 10 output logits.

Softmax cross-entropy minimizes the negative log probability of the assigned digit. Optax computes this stably from logits and integer targets. Adam updates only the classifier. The DDPM is neither loaded nor updated in this notebook.

In [ ]:
class DigitCNN(nn.Module):
    @nn.compact
    def __call__(self, pixels):
        x = nn.relu(nn.Conv(32, (3, 3), padding="SAME")(pixels))
        x = nn.max_pool(x, (2, 2), strides=(2, 2))
        x = nn.relu(nn.Conv(64, (3, 3), padding="SAME")(x))
        x = nn.max_pool(x, (2, 2), strides=(2, 2))
        x = x.reshape((x.shape[0], -1))
        x = nn.relu(nn.Dense(128)(x))
        return nn.Dense(10)(x)

SEED = 0
EPOCHS = 20
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
model = DigitCNN()
params = model.init(jax.random.key(SEED), jnp.zeros((1, 28, 28, 1)))["params"]
state = train_state.TrainState.create(
    apply_fn=model.apply, params=params, tx=optax.adam(LEARNING_RATE)
)

@jax.jit
def classifier_step(state, pixels, targets):
    def loss_fn(params):
        logits = state.apply_fn({"params": params}, pixels)
        return optax.softmax_cross_entropy_with_integer_labels(logits, targets).mean()
    loss, gradients = jax.value_and_grad(loss_fn)(state.params)
    return state.apply_gradients(grads=gradients), loss

@jax.jit
def predict(params, pixels):
    return model.apply({"params": params}, pixels).argmax(axis=-1)

## 4. Train on all generated images
Use 20 epochs, batch size 128, and Adam learning rate 0.001. Each epoch shuffles the data reproducibly and visits every example, including the final partial batch. Only each batch is placed on the active JAX device.

There is intentionally no train/test split. Rerun the model-initialization cell before repeating training from scratch. Epoch losses are weighted by batch size and saved to a flushed JSONL log. Re-running this cell replaces the previous log.

In [ ]:
rng = np.random.default_rng(SEED)
history = []
started = time.monotonic()
with (RESULT_DIR / "training.jsonl").open("w", buffering=1) as log:
    for epoch in range(EPOCHS):
        order = rng.permutation(len(images))
        total_loss = 0.0
        for start in range(0, len(order), BATCH_SIZE):
            indices = order[start:start + BATCH_SIZE]
            state, loss = classifier_step(
                state, jax.device_put(images[indices]), jax.device_put(y_target[indices])
            )
            total_loss += float(loss) * len(indices)
        record = {"epoch": epoch + 1, "loss": total_loss / len(images)}
        assert np.isfinite(record["loss"])
        history.append(record)
        log.write(json.dumps(record) + "\n")
        print(f"Epoch {epoch + 1:02d}/{EPOCHS}: loss={record['loss']:.5f}", flush=True)
print(f"Training duration: {time.monotonic() - started:.1f} seconds")

## 5. In-sample accuracy and confusion matrix
Evaluate the final CNN on exactly the same images used to update its parameters. Accuracy is the fraction whose predicted digit equals the conditioning target.

**Rows are conditioning labels; columns are predictions.** Entry (3,5) counts images requested as digit 3 that the classifier predicts as 5. Each row sums to 1,000. Diagonal counts are agreements; off-diagonal counts are disagreements. Disagreements alone cannot distinguish classifier errors from generator errors.

A row-normalized matrix reports fractions within each requested class. Save the counts, normalized matrix, predictions, metric JSON, and figure for the advisor. No accuracy is prefilled: execute training and this cell to calculate it.

In [ ]:
predicted = np.concatenate([
    np.asarray(predict(state.params, jax.device_put(images[start:start + BATCH_SIZE])))
    for start in range(0, len(images), BATCH_SIZE)
])
accuracy = float(np.mean(predicted == y_target))
confusion = np.zeros((10, 10), dtype=np.int64)
np.add.at(confusion, (y_target, predicted), 1)
assert confusion.sum() == len(images)
assert np.array_equal(confusion.sum(axis=1), np.bincount(y_target, minlength=10))
normalized = confusion / confusion.sum(axis=1, keepdims=True)

print(f"In-sample accuracy: {accuracy:.2%} ({np.trace(confusion)}/{len(images)})")
fig, ax = plt.subplots(figsize=(9, 8))
plot = ax.imshow(confusion, cmap="Blues")
for row in range(10):
    for column in range(10):
        value = confusion[row, column]
        ax.text(column, row, str(value), ha="center", va="center",
                color="white" if value > confusion.max() / 2 else "black")
ax.set(xticks=range(10), yticks=range(10), xlabel="Predicted digit",
       ylabel="Conditioning digit (target)",
       title=f"In-sample confusion matrix — accuracy {accuracy:.2%}")
fig.colorbar(plot, ax=ax, label="Number of images")
fig.tight_layout()
fig.savefig(RESULT_DIR / "confusion_matrix.png", dpi=160)
plt.show()

np.save(RESULT_DIR / "predictions.npy", predicted)
np.save(RESULT_DIR / "confusion_matrix.npy", confusion)
np.save(RESULT_DIR / "confusion_matrix_normalized.npy", normalized)
metrics = {
    "in_sample_accuracy": accuracy, "count": len(images), "epochs": EPOCHS,
    "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "seed": SEED,
    "dataset": str(DATA_DIR), "ddpm_checkpoint": metadata["checkpoint"],
    "checkpoint_sha256": metadata["checkpoint_sha256"],
    "evaluation": "same images used for training; no held-out evaluation",
}
(RESULT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
print("Saved evaluation:", RESULT_DIR)

## 6. Research interpretation
Report this percentage as **CNN in-sample accuracy on 10,000 conditional DDPM samples**, with the classifier settings and confusion matrix. A strong diagonal indicates that this CNN fits the assigned conditions. It does not establish performance on unseen samples, real MNIST, or independent semantic correctness. An independently trained real-MNIST classifier would provide a complementary evaluation.